In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
from scipy import sparse
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, log_loss





# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# MLFlow Initialization

In [22]:
%pip install -q dagshub mlflow

Note: you may need to restart the kernel to use updated packages.


In [23]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-2-IEEE-CIS-Fraud-Detection', mlflow=True)

Initialized MLflow to track repo "izere23/Assignment-2-IEEE-CIS-Fraud-Detection"

Repository izere23/Assignment-2-IEEE-CIS-Fraud-Detection initialized!

In [24]:
import mlflow

# Data Loading

In [25]:
BASE_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
train_transaction = pd.read_csv(BASE_PATH + "train_transaction.csv")
train_identity = pd.read_csv(BASE_PATH + "train_identity.csv")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)
print(train["isFraud"].value_counts(normalize=True))

(590540, 434)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [26]:
X = train.drop(columns=["isFraud"])
y = train["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(472432, 433) (118108, 433)
0.03498916246147594 0.0349933958749619


# Cleaning

## High Missing Features

In [27]:
class HighMissingDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold].tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

## High Dominance Columns (Not Using)

In [28]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class HighDominanceDropper(BaseEstimator, TransformerMixin):
    def __init__(self, dominance_threshold):
        self.dominance_threshold = dominance_threshold

    def fit(self, X, y=None):
        X = X.copy()

        dominance = X.apply(
            lambda col: col.value_counts(normalize=True, dropna=False).iloc[0]
            if col.value_counts(dropna=False).shape[0] > 0 else 1
        )

        self.cols_to_drop_ = dominance[
            dominance > self.dominance_threshold
        ].index.tolist()

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

### Cleaning Pipeline

In [29]:
cleaning_pipeline = Pipeline([
    ("high_missing_columns", HighMissingDropper(threshold=0.98))
])

# Preprocessing

In [30]:
class AutoPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

        num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        self.preprocessor_ = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])

        self.preprocessor_.fit(X, y)
        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

# Feature Engineering

In [31]:
class MissingCountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X["missing_count"] = X.isna().sum(axis=1)
        return X

In [32]:
class ZeroCountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X["zero_count"] = (X == 0).sum(axis=1)
        return X

In [33]:
class TransactionTimeFeaturesAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionDT" in X.columns:
            X["transaction_hour"] = (X["TransactionDT"] // 3600) % 24
            X["transaction_day"] = (X["TransactionDT"] // (3600 * 24)) % 7
            X["transaction_week"] = X["TransactionDT"] // (3600 * 24 * 7)

        return X

In [34]:
class UIDAmountDeviation(BaseEstimator, TransformerMixin):
    def __init__(self, uid_cols=["card1", "addr1"], amt_col="TransactionAmt"):
        self.uid_cols = uid_cols
        self.amt_col = amt_col

    def fit(self, X, y=None):
        X = X.copy()
        self.uid_name_ = "_".join(self.uid_cols)

        X[self.uid_name_] = X[self.uid_cols].astype(str).agg("_".join, axis=1)

        self.uid_mean_ = X.groupby(self.uid_name_)[self.amt_col].mean()

        return self

    def transform(self, X):
        X = X.copy()

        X[self.uid_name_] = X[self.uid_cols].astype(str).agg("_".join, axis=1)

        X["uid_amt_mean"] = X[self.uid_name_].map(self.uid_mean_)

        X["uid_amt_diff"] = X[self.amt_col] - X["uid_amt_mean"]
        X["uid_amt_ratio"] = X[self.amt_col] / (X["uid_amt_mean"] + 1e-5)

        return X

In [35]:
feature_engineering_pipeline = Pipeline([
    ("uid_amt_dev", UIDAmountDeviation())
])

# Feature Selection

## RFE

In [36]:
class RFESelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_features_to_select=100, step=0.1):
        self.n_features_to_select = n_features_to_select
        self.step = step
        self.selector_ = None

    def fit(self, X, y=None):
        estimator = DecisionTreeClassifier(
            max_depth=12,
            min_samples_split=18,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=42
        )

        self.selector_ = RFE(
            estimator=estimator,
            n_features_to_select=self.n_features_to_select,
            step=self.step
        )

        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        return self.selector_.transform(X)

## Correlation Filter

In [37]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.to_keep_ = None

    def fit(self, X, y=None):
        if sparse.issparse(X):
            X = X.toarray()   # okay for 434 features

        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()

        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        to_drop = [
            col for col in upper.columns
            if any(upper[col] > self.threshold)
        ]

        self.to_keep_ = [
            col for col in X_df.columns
            if col not in to_drop
        ]

        return self

    def transform(self, X):
        if sparse.issparse(X):
            X = X.toarray()

        return X[:, self.to_keep_]

## Log Adder

In [38]:
class LogAmountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])

        return X

# Decision Tree Pipeline

In [39]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_split=18,
    min_samples_leaf=8,
    class_weight="balanced",
    random_state=42
)

pipeline = Pipeline([
    ("drop_high_missing", HighMissingDropper(threshold=0.98)),
    ("preprocessing", AutoPreprocessor()),
    ("model", model)
])

# Training

In [42]:
mlflow.set_experiment("Decision_Tree_Training")

with mlflow.start_run(run_name="DT_FINAL_RUN"):

    threshold = 0.5
    
    pipeline.fit(X_train, y_train)

    train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
    val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

    val_pred = (val_pred_proba >= threshold).astype(int)

    train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    val_roc_auc = roc_auc_score(y_val, val_pred_proba)

    train_log_loss = log_loss(y_train, train_pred_proba)
    val_log_loss = log_loss(y_val, val_pred_proba)

    overfit_gap = train_roc_auc - val_roc_auc
    overfit_loss_gap = val_log_loss - train_log_loss

    precision = precision_score(y_val, val_pred)
    recall = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)

    mlflow.log_param("model", "decision_tree")
    mlflow.log_param("baseline_type", "raw_data_minimal_preprocessing")
    
    mlflow.log_param("numeric_imputation", "median")
    mlflow.log_param("categorical_imputation", "most_frequent")
    mlflow.log_param("encoding", "OHE")
    
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("min_samples_split", 18)
    mlflow.log_param("min_samples_leaf", 8)
    
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    
    mlflow.log_param("high_missing_threshold", 0.98)
    
    mlflow.log_param("feature_engineering", "False")
    mlflow.log_param("feature_selection", "False")
    
    mlflow.log_metric("train_roc_auc", train_roc_auc)
    mlflow.log_metric("val_roc_auc", val_roc_auc)
    mlflow.log_metric("overfit_gap", overfit_gap)

    mlflow.log_metric("train_log_loss", train_log_loss)
    mlflow.log_metric("val_log_loss", val_log_loss)
    mlflow.log_metric("overfit_loss_gap", overfit_loss_gap)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="decision_tree"
    )

    print("Train ROC-AUC:", train_roc_auc)
    print("Val ROC-AUC:", val_roc_auc)
    print("Overfit Gap:", overfit_gap)
    print("Train Log Loss:", train_log_loss)
    print("Val Log Loss:", val_log_loss)
    print("Overfit Loss Gap:", overfit_loss_gap)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

2026/05/06 15:05:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:05:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train ROC-AUC: 0.9104721033101745
Val ROC-AUC: 0.8705918567363186
Overfit Gap: 0.039880246573855915
Train Log Loss: 0.35363619276210195
Val Log Loss: 0.4051859228295417
Overfit Loss Gap: 0.051549730067439736
Precision: 0.17418282548476455
Recall: 0.7607065085894024
F1: 0.28346030744263623
🏃 View run DT_FINAL_RUN at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/23e816a6103f485696a311cff7dcc040
🧪 View experiment at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
